# 마지막 7일 test · 검증 없음 · 10시드 평균

직전 seed44 개발용 노트북을 대체합니다. **L4 GPU**를 연결하고 위에서부터 실행하세요. 이 노트북은 선택한 데이터 하나만 실행합니다.

| 데이터 | 그래프/BPR 학습 | 최종 test |
|---|---|---|
| Dunnhumby | DAY 1–704 전체 | DAY 705–711 |
| H&M | 2018-09-20–2020-09-15 전체 | 2020-09-16–22 |

- validation·holdout 없음. 학습량 300epoch 고정, 마지막 checkpoint만 test 평가. test로 epoch/설정 선택·조기종료하지 않습니다.
- 시드 42–51을 네 모형 모두 동일하게 사용: **M1 → M3+M4-B → 새 M2 → M2+M3+M4-B**, 각자 무작위 초기화부터 공동학습.
- **데이터 하나당 40회 학습(12,000epoch)**입니다. 두 데이터를 모두 실행하면 80회입니다. 기존 분할의 결과/checkpoint는 재사용하지 않습니다. H&M은 기본 실행되지 않습니다.
- 전체 앞기간은 그래프·BPR 학습에 사용하고, CLV N/V/q_C 계산의 관찰창은 기존 **학습 종료 전 365일**을 유지합니다. 두 기간은 서로 다릅니다.
- 신규상품 과업·MIN_ITEM_INTER=1·uniform K1 유지. 학습에서 본 사용자–상품 쌍은 평가 정답과 후보에서 제외합니다.
- 마지막 주는 초기 연구에서 이미 확인한 기간입니다. **처음 보는 test의 독립 확증으로 부르지 않습니다.** 열 시드는 초기화/학습 변동을 보며 test 재사용 편향을 없애지 않습니다.

## 모형·판정은 실행 전 고정

M2: 사용자 경제표현 q_C·[T0+(2q_N−1)TN]·b(q_V), 상품 경제표현 b(상품 구매금액 백분위). T0=I, TN=0으로 시작하고 ID와 같은 optimizer/BPR로 함께 학습합니다. N·V 모두 유지하고 경제표현은 그래프에서 전파하지 않습니다. 최종 내적 전에 ID표현과 연결합니다.

M2 단독은 binary/plain BPR. 결합은 중심화 **N/V 두 축 M3** + **split N/V M4-B**(무효입력 행 raw=1, 학습행 평균 정규화). 동일 수식을 두 데이터에 적용하되 학습구간 백분위와 M3 목표 가중치 CV=.20으로 보정합니다. T0/TN은 각 데이터에서 학습됩니다. ρ=.25·RBF폭=.25는 고정이며 별도 자동탐색은 없습니다.

ID64·2층·lr=.0005·ID L2=.001. batch는 Dunnhumby8192/H&M131072입니다. 자유 행렬에 추가 규제/크기제약은 없으므로 ρ는 경제점수 크기의 상한이 아닙니다.

10시드 평균에서 전체결합의 **전체 사용자 가격·구매금액 가중 적중값@10 및 가중 NDCG@10이 각각 M1과 M3+M4-B를 모두 초과**, 전체 Recall/NDCG@10·20·50 평균 각각 M1의99% 이상인지 보고합니다. M2 단독 미달은 결합의 학습/판정을 막지 않습니다. 시드별 불리한 결과·모든 세그먼트·노출 지표도 보존합니다. 유의성·CLV 배정 귀속은 주장하지 않습니다.

로컬 소형 CPU 분할·공동학습·checkpoint재개·집계 검사만 완료했습니다. 실제 전체 데이터/GPU 학습은 미실행입니다.

## 1. Drive와 고정 코드 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json, torch
SOURCE_COMMIT = 'ad45a16d86ad05b9f0d27aab24c8d4fe468e7ee4'
REPO = Path('/content/clv-lastweek-test10-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
assert torch.cuda.is_available(), 'GPU 런타임을 연결하세요. L4 권장.'
sys.path.insert(0, str(REPO))
print('source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0))


## 2. 데이터 선택·준비 — 아직 학습하지 않음

기본은 Dunnhumby입니다. H&M 실행을 선택하면 DATASET만 'hm'으로 설정하세요. 데이터 변경 시 런타임을 다시 시작해 위에서부터 실행합니다. 원본 기간·최종 분할·train-only 가중치 사전점검 불일치 시 학습 전에 중단합니다.

In [ ]:
import clv_m2_m5_lastweek_test10 as screen
DATASET = 'dunnhumby'  # @param ["dunnhumby", "hm"]
cfg = screen.configure(DATASET)
print('선택 데이터:', cfg.dataset, '| 시드:', cfg.seeds, '| 4모형 × 10시드 = 40회 학습')
prepared = screen.prepare(cfg)
assert set(prepared['data']['splits']) == {'test'}
print('실행/자동재개 경로:', prepared['run_dir'])


## 3. 40회 순차 학습

매 epoch optimizer·난수상태를 저장하고 중단 후 재실행하면 마지막 완료 epoch에서 재개합니다. 이번 분할에서 완료한 모델만 재사용합니다. 25epoch마다 loss가 출력되며 그 사이 진행상태는 실행폴더/progress에 저장됩니다. 학습 중 test 성능은 계산하지 않고 각 모델의300epoch에서만 평가합니다. 전체 완료 전 10시드 최종 판정은 생성하지 않습니다.

In [ ]:
result = screen.run(cfg, prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


## 4. 10시드 평균·표준편차 및 대응 비교

상대 변화율은 시드별 비율 평균이 아닌 (모형 평균/기준 평균−1)×100입니다. 대응 차이 평균·표준편차와 개선 시드 수도 함께 출력합니다. 전체142지표/시드별 원본은 ZIP에 포함됩니다.

In [ ]:
metrics = [f'{m}@{k}' for k in (10,20,50) for m in ('recall','ndcg','price_purchase_amount_weighted_hit','vndcg')]
labels = {screen.M1:'M1', screen.BASE:'M3+M4-B', screen.M2:'새 M2', screen.FULL:'M2+M3+M4-B'}
summary = result['summary'].copy()
summary['model_id'] = summary.model_id.map(labels)
for stat in ('mean', 'std'):
    print('\n10시드 ' + stat)
    print(summary[summary.metric.isin(metrics)].pivot(index='metric', columns='model_id', values=stat).to_string())
comparison = result['comparison'].copy()
comparison['model_id'] = comparison.model_id.map(labels)
comparison['reference'] = comparison.reference.map(labels)
print('\n대응 비교: 변화율 %, 차이 평균·표준편차, 개선 시드 수')
print(comparison[comparison.metric.isin(metrics)].to_string(index=False))
print('\n시드별 절대지표 — 미달 결과 포함')
print(result['absolute'][['model_id','seed']+metrics].to_string(index=False))
print('\nN/V 작동 진단')
print(result['diagnostics'].to_string(index=False))
print(json.dumps(result['paths'], ensure_ascii=False, indent=2))


## 5. 전체 결과 ZIP 다운로드

absolute/summary/comparison/paired_comparison/diagnostics 및 전체 설정·판정·학습이력을 포함한 result.json을 묶습니다. 큰 checkpoint는 Drive에 남습니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
zip_path = Path('/content') / f'm2_m5_lastweek_test10_{cfg.dataset}_results.zip'
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for name, path in result['paths'].items():
        path = Path(path)
        archive.write(path, arcname=path.name)
    archive.write(prepared['run_dir'] / 'protocol.json', arcname='protocol.json')
print(zip_path)
files.download(str(zip_path))
